In [5]:
# ============================================================
# REPOSITORY SETUP — YOUR GitHub repository
# ============================================================

from pathlib import Path
import shutil
import urllib.request
import zipfile

ZIP_URL = "https://codeload.github.com/Imvixh/flyrank-ml-internship/zip/refs/heads/main"

ROOT = Path("/content/flyrank-ml-internship")
ZIP_PATH = Path("/content/flyrank-ml-internship-main.zip")

if ROOT.exists():
    shutil.rmtree(ROOT)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

print("Downloading YOUR GitHub repository...")
urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)

print("Extracting repository...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

EXTRACTED = Path("/content/flyrank-ml-internship-main")

if ROOT.exists():
    shutil.rmtree(ROOT)

EXTRACTED.rename(ROOT)
ZIP_PATH.unlink(missing_ok=True)

DATA = ROOT / "data/raw/content_refresh_anonymized.csv"

print("\n" + "=" * 55)
print("REPOSITORY CHECK")
print("=" * 55)

print("Repository exists:", ROOT.exists())
print("Dataset exists:", DATA.exists())
print("Repository:", ROOT)
print("Dataset:", DATA)

if not ROOT.exists():
    raise FileNotFoundError("Repository could not be prepared.")

if not DATA.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA}")

print("\n✓ YOUR repository ready")
print("✓ Dataset ready")

Extracting repository...

REPOSITORY CHECK
Repository exists: True
Dataset exists: True
Repository: /content/flyrank-ml-internship
Dataset: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

✓ YOUR repository ready
✓ Dataset ready


# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution review

The key numeric signals have different scales and several are likely to be heavy-tailed. Search volume, impressions, clicks, and sessions can vary substantially across pages, so averages alone may hide the typical page. I will use median, selected percentiles, and missingness to describe the distributions before interpreting the signals.

In [6]:
# ============================================================
# SECTION 1 — DISTRIBUTIONS
# ============================================================

import pandas as pd
import numpy as np

df = pd.read_csv(DATA)

key_fields = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
]

available_fields = [c for c in key_fields if c in df.columns]

distribution_rows = []

for col in available_fields:
    s = pd.to_numeric(df[col], errors="coerce")

    distribution_rows.append({
        "field": col,
        "missing_pct": round(s.isna().mean() * 100, 2),
        "median": s.median(),
        "p25": s.quantile(0.25),
        "p75": s.quantile(0.75),
        "p95": s.quantile(0.95),
        "max": s.max(),
    })

distribution_table = pd.DataFrame(distribution_rows)

display(distribution_table)

print("\nRows:", len(df))
print("Fields audited:", len(available_fields))

print("\n✓ Key-field distributions inspected")
print("✓ Missingness inspected")
print("✓ Heavy-tail indicators reviewed using P95 and maximum")

,field,missing_pct,median,p25,p75,p95,max
0,search_volume,8.23,10.00,0.0,20.00,390.00,74000.0
1,impressions_90d,0.00,731.00,81.0,3615.25,22996.50,517715.0
2,clicks_90d,0.00,1.00,0.0,7.00,69.05,4178.0
3,sessions_90d,0.00,7.00,2.0,27.00,166.00,4345.0
4,ctr,0.00,0.07,0.0,0.29,1.09,100.0
5,avg_position,0.00,10.80,6.2,22.30,48.20,245.0
6,engagement_rate,0.00,0.00,0.0,1.35,12.50,100.0
7,scroll_rate,0.42,5.00,0.0,23.53,100.00,300.0
8,content_age_days,0.00,236.00,132.0,333.00,487.00,564.0
9,days_since_last_update,0.00,20.00,20.0,104.00,104.00,373.0



Rows: 30000
Fields audited: 10

✓ Key-field distributions inspected
✓ Missingness inspected
✓ Heavy-tail indicators reviewed using P95 and maximum


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

**Signal #1 — Trend direction:**  
Pages with an observed `down` trend should have a higher average negative trend percentage than pages that are not down.  
**Verdict:** CONFIRMED if the data supports this directional difference.

**Signal #2 — CTR and ranking position:**  
Pages that rank relatively well but have low CTR may represent a review opportunity.  
**Verdict:** CONFIRMED if low-CTR pages with visible positions show the expected pattern.

**Signal #3 — Content age:**  
Older content may be more likely to require review than newer content.  
**Verdict:** CONFIRMED, OPPOSITE, MIXED, or FALSE based on the observed data rather than assumption.

These are directional associations in the observed dataset, not causal claims.

In [7]:
# ============================================================
# SECTION 2 — SIGNAL TESTS #1 / #2 / #3
# ============================================================

# Signal 1: Trend direction vs trend percentage
print("=== SIGNAL #1: TREND DIRECTION ===")

trend_test = (
    df.groupby("trend_direction", dropna=False)["trend_pct"]
      .agg(["count", "mean", "median"])
      .sort_values("mean")
)

display(trend_test)

down_mean = df.loc[
    df["trend_direction"].eq("down"), "trend_pct"
].mean()

non_down_mean = df.loc[
    ~df["trend_direction"].eq("down"), "trend_pct"
].mean()

if down_mean < non_down_mean:
    verdict_1 = "CONFIRMED"
else:
    verdict_1 = "OPPOSITE"

print("Down mean trend_pct:", round(down_mean, 2))
print("Non-down mean trend_pct:", round(non_down_mean, 2))
print("Verdict:", verdict_1)


# Signal 2: Low CTR with visible ranking
print("\n=== SIGNAL #2: LOW CTR + VISIBLE POSITION ===")

visible = df[
    pd.to_numeric(df["avg_position"], errors="coerce") <= 10
].copy()

visible["low_ctr"] = (
    pd.to_numeric(visible["ctr"], errors="coerce") < 1
)

ctr_test = visible.groupby("low_ctr")["ctr"].agg(
    ["count", "mean", "median"]
)

display(ctr_test)

low_ctr_down_rate = (
    visible.loc[visible["low_ctr"], "trend_direction"]
    .eq("down")
    .mean()
)

normal_ctr_down_rate = (
    visible.loc[~visible["low_ctr"], "trend_direction"]
    .eq("down")
    .mean()
)

print("Low-CTR down rate:", round(low_ctr_down_rate, 3))
print("Other-CTR down rate:", round(normal_ctr_down_rate, 3))

if low_ctr_down_rate > normal_ctr_down_rate:
    verdict_2 = "CONFIRMED"
elif low_ctr_down_rate < normal_ctr_down_rate:
    verdict_2 = "OPPOSITE"
else:
    verdict_2 = "MIXED"

print("Verdict:", verdict_2)


# Signal 3: Content age
print("\n=== SIGNAL #3: CONTENT AGE ===")

df["age_group"] = pd.cut(
    pd.to_numeric(df["content_age_days"], errors="coerce"),
    bins=[-1, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"]
)

age_test = (
    df.groupby("age_group", observed=False)["trend_direction"]
      .apply(lambda s: s.eq("down").mean())
      .to_frame("down_rate")
)

display(age_test)

age_rates = age_test["down_rate"].dropna()

if len(age_rates) >= 2:
    if age_rates.iloc[-1] > age_rates.iloc[0]:
        verdict_3 = "CONFIRMED"
    elif age_rates.iloc[-1] < age_rates.iloc[0]:
        verdict_3 = "OPPOSITE"
    else:
        verdict_3 = "MIXED"
else:
    verdict_3 = "FALSE"

print("Verdict:", verdict_3)


print("\n=== SIGNAL TEST SUMMARY ===")
print("Signal #1:", verdict_1)
print("Signal #2:", verdict_2)
print("Signal #3:", verdict_3)

=== SIGNAL #1: TREND DIRECTION ===


,count,mean,median
trend_direction,,,
down,16262,-58.113830,-55.60
stable,5962,-3.185944,-3.80
up,4388,190.673997,62.55
flat,0,NaN,NaN
new,0,NaN,NaN


Down mean trend_pct: -58.11
Non-down mean trend_pct: 79.0
Verdict: CONFIRMED

=== SIGNAL #2: LOW CTR + VISIBLE POSITION ===


,count,mean,median
low_ctr,,,
False,1162,7.616661,2.08
True,13026,0.177681,0.08


Low-CTR down rate: 0.523
Other-CTR down rate: 0.433
Verdict: CONFIRMED

=== SIGNAL #3: CONTENT AGE ===


,down_rate
age_group,
0-90,0.668699
91-180,0.625552
181-365,0.514866
365+,0.426258


Verdict: OPPOSITE

=== SIGNAL TEST SUMMARY ===
Signal #1: CONFIRMED
Signal #2: CONFIRMED
Signal #3: OPPOSITE


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test

The baseline uses `low_ctr_visible_page` as a reason to prioritize human review. The assumption is that a page can have a relatively visible search position while receiving weak click-through.

The test compares low-CTR visible pages with other visible pages and checks their observed trend and CTR. This can support the flag as a directional review signal, but it cannot establish that improving the page will cause higher CTR or traffic.

In [8]:
# ============================================================
# SECTION 3 — FLAG-LINKED TEST
# ============================================================

position_num = pd.to_numeric(df["avg_position"], errors="coerce")
ctr_num = pd.to_numeric(df["ctr"], errors="coerce")

visible = position_num <= 10
low_ctr_visible = visible & (ctr_num < 1)

flag_test = pd.DataFrame({
    "group": [
        "low_ctr_visible_page",
        "other_visible_pages"
    ],
    "rows": [
        low_ctr_visible.sum(),
        (visible & ~low_ctr_visible).sum()
    ],
    "mean_ctr": [
        ctr_num[low_ctr_visible].mean(),
        ctr_num[visible & ~low_ctr_visible].mean()
    ],
    "down_rate": [
        df.loc[low_ctr_visible, "trend_direction"].eq("down").mean(),
        df.loc[
            visible & ~low_ctr_visible,
            "trend_direction"
        ].eq("down").mean()
    ]
})

display(flag_test)

low_ctr_down = flag_test.loc[
    flag_test["group"] == "low_ctr_visible_page",
    "down_rate"
].iloc[0]

other_visible_down = flag_test.loc[
    flag_test["group"] == "other_visible_pages",
    "down_rate"
].iloc[0]

print("Low-CTR visible down rate:",
      round(low_ctr_down, 3))

print("Other visible down rate:",
      round(other_visible_down, 3))

if low_ctr_down > other_visible_down:
    flag_verdict = "CONFIRMED"
elif low_ctr_down < other_visible_down:
    flag_verdict = "OPPOSITE"
else:
    flag_verdict = "MIXED"

print("Flag verdict:", flag_verdict)

print("\n✓ Flag-linked assumption tested")

,group,rows,mean_ctr,down_rate
0,low_ctr_visible_page,13026,0.177681,0.523261
1,other_visible_pages,1162,7.616661,0.432874


Low-CTR visible down rate: 0.523
Other visible down rate: 0.433
Flag verdict: CONFIRMED

✓ Flag-linked assumption tested


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The signal audit shows which observed signals are useful for prioritizing content-review candidates, but the signals should be treated as directional rather than causal. A content team should give higher attention to pages with multiple supporting signals, such as observed decline, meaningful search demand, visible ranking position, and weak CTR.

The baseline flags should not be used as automatic refresh decisions. Human review is still required to check data completeness, search intent, seasonality, and whether the observed pattern is meaningful for the specific page.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check — VERIFIED

- [x] Every section above is filled — markdown thinking and supporting code are complete.
- [x] The notebook runs top to bottom with no errors — verified using Runtime → Run all.
- [x] No client names, URLs, or private queries are included.
- [x] Claims use careful words such as observed, measured, directional, and decision-support.
- [x] Key-field distributions were inspected.
- [x] Three signals were tested with explicit verdicts.
- [x] A flag-linked assumption was tested against the observed data.
- [x] Practical implications were stated without causal claims.
- [x] The completed notebook was committed to my repository under `work/notebooks/`.